# RunwayEye · antrenarea modelului YOLO (FOD) în Google Colab

Alternativă la `training/train_local.py`, pentru cine nu are placă video NVIDIA.
Modelul din repo (`models/best.pt`) e antrenat pe cele 31 de clase FOD-A (`bolt`, `nut`, `wrench`…), vezi `app/vision/types.py`.

1. **Runtime → Change runtime type → T4 GPU**.
2. Rulați celulele în ordine. Antrenarea durează ~30–60 de minute.
3. La final se descarcă `best.pt` și `metrics.json`. Copiați-le în `models/` din proiect.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow

## Setul de date

**Varianta B (recomandată, fără cont):** pe laptop, `python training/prepare_fod_a.py <folderul VOC2007 din FOD-A>` face `datasets/fod-a/yolo/` (împărțit train/val/test fără cadre vecine în seturi diferite). Arhivați **conținutul** folderului `datasets/fod-a/yolo` ca `.zip` și încărcați-l la celula următoare, cu `USE_ROBOFLOW_API = False`.

**Varianta A: din Roboflow** (de ex. poze cu macheta voastră). Cheia API în Colab: iconița 🔑 *Secrets* → `ROBOFLOW_API_KEY`. Numele claselor trebuie să fie clasele FOD-A sau grupele `surub`, `piulita`, `cheie`, `metal`, `plastic`, `piatra`.

In [ ]:
USE_ROBOFLOW_API = False
WORKSPACE = "workspace-ul-vostru"
PROJECT = "runwayeye-fod"
VERSION = 1

import glob, os, zipfile

if USE_ROBOFLOW_API:
    from google.colab import userdata
    from roboflow import Roboflow

    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
    DATA = os.path.join(dataset.location, "data.yaml")
else:
    from google.colab import files

    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("dataset")
    DATA = os.path.abspath(glob.glob("dataset/**/data.yaml", recursive=True)[0])

print("data.yaml:", DATA)
print(open(DATA).read())

In [ ]:
# Verificare: numele claselor trebuie să fie cele cunoscute de aplicație (app/vision/types.py).
import yaml

FOD_A = {"adjustable_clamp", "adjustable_wrench", "battery", "bolt", "bolt_nut_set", "bolt_washer", "clamp_part",
         "cutter", "fuel_cap", "hammer", "hose", "label", "luggage_part", "luggage_tag", "metal_part", "metal_sheet",
         "nail", "nut", "paint_chip", "pen", "plastic_part", "pliers", "rock", "screw", "screwdriver", "soda_can",
         "tape", "washer", "wire", "wood", "wrench"}
COARSE = {"surub", "piulita", "cheie", "metal", "plastic", "piatra"}
names = yaml.safe_load(open(DATA))["names"]
names = set(names.values() if isinstance(names, dict) else names)
print("clase:", sorted(names))
unknown = names - FOD_A - COARSE
if unknown:
    print("ATENȚIE: aplicația nu cunoaște", sorted(unknown), "- le afișează cu numele din model și risc «de verificat»")

## Antrenare

Model nano. FOD-A are imagini de 300×300, deci `imgsz=320`; pentru poze proprii mai mari, puneți 640.

In [ ]:
from ultralytics import YOLO

EPOCHS = 30
IMGSZ = 320
model = YOLO("yolo11n.pt")  # pre-antrenat pe COCO; învățăm doar clasele noastre
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=32,
    patience=10,
    cos_lr=True,
    flipud=0.5,
    fliplr=0.5,
    scale=0.6,
    project="runs",
    name="fod",
    exist_ok=True,
)

## Rezultate (pentru slide)

mAP pe setul de **test** (imagini pe care modelul nu le-a văzut la antrenare), sau pe validare dacă setul nu are test.

In [ ]:
import json

best = YOLO("runs/fod/weights/best.pt")
split = "test" if "test" in yaml.safe_load(open(DATA)) else "val"
m = best.val(data=DATA, imgsz=IMGSZ, split=split)
metrics = {
    "split": split,
    "mAP50": round(float(m.box.map50), 4),
    "mAP50_95": round(float(m.box.map), 4),
    "precision": round(float(m.box.mp), 4),
    "recall": round(float(m.box.mr), 4),
    "per_class": {best.names[int(c)]: {"mAP50": round(float(m.box.ap50[i]), 4), "mAP50_95": round(float(m.box.ap[i]), 4)}
                  for i, c in enumerate(m.box.ap_class_index)},
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "base_model": "yolo11n.pt",
}
json.dump(metrics, open("metrics.json", "w"), indent=2)
print(json.dumps(metrics, indent=2))

In [ ]:
# Imagini utile pentru prezentare: matricea de confuzie și curbele de antrenare.
from IPython.display import Image, display

for f in ["confusion_matrix_normalized.png", "results.png"]:
    p = f"runs/fod/{f}"
    if os.path.exists(p):
        display(Image(p, width=700))

In [ ]:
from google.colab import files

files.download("runs/fod/weights/best.pt")
files.download("metrics.json")
!zip -qj fod_plots.zip runs/fod/*.png
files.download("fod_plots.zip")